# Messages, streaming, batching, and Memory

In [ ]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage

# Load GROQ_API_KEY from the local .env file.
load_dotenv()

groq_model = init_chat_model("openai/gpt-oss-120b", model_provider="groq")


## Messages: text plus a label

In [ ]:
# A HumanMessage carries both the text and the "human" label.
human_question = HumanMessage("How do I create a vegan chicken sandwich?")
print("Label:", human_question.type)
print("Text:", human_question.content)


In [ ]:
# Without labels, the model receives one blob of instructions and question text.
plain_text_response = groq_model.invoke(
    "You are a mad street chef. How do I create a vegan chicken sandwich?",
    max_tokens=200,  # limit the response length
)
print(plain_text_response.text)


In [ ]:
# With labels, rules and the user's question are separate messages.
message_response = groq_model.invoke(
    [
        SystemMessage("You are a mad street chef."),  # application rules
        HumanMessage("How do I create a vegan chicken sandwich?"),  # user question
    ],
    max_tokens=200,  # limit the response length
)
print(message_response.text)

In [ ]:
print("~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~")
# An AIMessage can carry an earlier reply into the next model call.
conversation_with_history = [
    SystemMessage("You are a mad street chef."),
    HumanMessage("How do I create a vegan chicken sandwich?"),
    AIMessage(
        "First, you'll need to find a vegan chicken. And fry it "
    ),  # earlier model reply
    # note we are assuming this is the model's response to the previous question.
    HumanMessage("can I create a bread from it?"),
]

history_response = groq_model.invoke(conversation_with_history, max_tokens=200)
print(history_response.text)

## Stream a response

In [ ]:
stream_prompt = "In a funny way explain why burgers are better than salads in 10 sentences."

for chunk in groq_model.stream(stream_prompt):
    # Each chunk contains the next available piece of the model response.
    print(chunk.text, end="", flush=True)
print()


# Print chunks as they arrive.

## Batch independent requests

In [ ]:
batch_prompts = [
    "Give one use for a low temperature setting.",
    "Give one use for a high temperature setting.",
    "Define an LLM in one sentence.",
]

batch_responses = groq_model.batch(batch_prompts)
for prompt, batch_response in zip(batch_prompts, batch_responses):
    print(f"Prompt: {prompt}")
    print(f"Answer: {batch_response.text}")
    print()


## Keep short-term memory yourself

In [ ]:
conversation = [
    SystemMessage("You are a concise assistant for a Python course."),
    HumanMessage("My name is Ada. I am learning about LLMs."),
]

first_response = groq_model.invoke(conversation)
conversation.append(first_response)
conversation.append(HumanMessage("What is my name and what am I learning?"))

second_response = groq_model.invoke(conversation)
print(second_response.text)
